In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1. 벡터 차원 정의: 사용 중인 임베딩 모델이 생성하는 결과물의 길이(차원)를 계산합니다.
# 임베딩 모델(embeddings)은 사전에 정의되어 있어야 합니다.
embedding_dim = len(embeddings.embed_query("hello world"))

# 2. FAISS 인덱스 초기화: 벡터 간의 거리 계산 방식을 결정합니다.
# IndexFlatL2: 유클리드 거리($L2$ distance)를 계산하는 가장 정확한 완전 탐색(Brute-force) 방식입니다.
index = faiss.IndexFlatL2(embedding_dim)

# 3. LangChain FAISS 객체 조립: 검색 엔진(index)과 원본 데이터 저장소(docstore)를 결합합니다.
vector_store = FAISS(
    embedding_function=embeddings,  # 텍스트를 벡터로 변환할 함수
    index=index,                   # 유사도 검색을 수행할 FAISS 인덱스
    docstore=InMemoryDocstore(),   # 실제 텍스트 내용과 메타데이터를 담을 메모리 저장소
    index_to_docstore_id={},       # 인덱스 번호와 저장소 ID 간의 매핑 테이블(초기화 시 빈 값)
)

# 4. 데이터 준비: 검색 대상이 될 문서 객체들을 생성합니다.
documents = [
    Document(page_content="컴퓨터 공학은 하드웨어와 소프트웨어를 연구하는 학문입니다.", metadata={"source": "edu"}),
    Document(page_content="인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다.", metadata={"source": "tech"}),
    Document(page_content="고양이는 귀여운 동물이며 많은 사람들이 반려 동물로 키웁니다.", metadata={"source": "pets"}),
    Document(page_content="파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다.", metadata={"source": "tech"}),
]

# 5. 데이터 적재: 문서를 임베딩하여 벡터화한 뒤 FAISS 인덱스에 추가합니다.
# 내부적으로 임베딩 생성 -> 인덱스 저장 -> docstore 저장이 동시에 수행됩니다.
vector_store.add_documents(documents=documents)

# 6. 유사도 검색(Top-K): 질문과 의미적으로 가장 가까운 문서 k개를 추출합니다.
# query = "머신러닝과 AI 기술에 대해 알려줘"
query = "야생 고양이의 습성을 알려줘"
results = vector_store.similarity_search(query, k=2)

print(f"--- [검색 질의]: {query} ---")
for i, doc in enumerate(results):
    print(f"결과 {i+1}: {doc.page_content} (출처: {doc.metadata['source']})")

# 7. 점수 포함 검색: 거리 값(Distance)을 포함하여 검색 결과의 신뢰도를 확인합니다.
# IndexFlatL2를 사용하므로 점수(Score)는 거리를 의미하며, 0에 가까울수록 유사도가 높습니다.
results_with_score = vector_store.similarity_search_with_score(query, k=4)

print("\n--- [점수 포함 검색 결과] ---")
for doc, score in results_with_score:
    print(f"거리(Score): {score:.4f} | 내용: {doc.page_content}")

--- [검색 질의]: 머신러닝과 AI 기술에 대해 알려줘 ---
결과 1: 인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다. (출처: tech)
결과 2: 파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다. (출처: tech)

--- [점수 포함 검색 결과] ---
거리(Score): 0.4197 | 내용: 인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다.
거리(Score): 0.6014 | 내용: 파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다.
거리(Score): 0.8231 | 내용: 컴퓨터 공학은 하드웨어와 소프트웨어를 연구하는 학문입니다.
거리(Score): 0.9514 | 내용: 고양이는 귀여운 동물이며 많은 사람들이 반려 동물로 키웁니다.
